# Coleta de Manuais de Campanha — BDEx

Baixa automaticamente os **Manuais de Campanha** do Exército Brasileiro a partir da
Biblioteca Digital do Exército ([BDEx](https://bdex.eb.mil.br/jspui/)), alimentando a
pasta `Arquivos/` que o `scripts/pipeline.py` consome.

**Fluxo:**
1. Lê o facet *Tipo de Documento* da busca e confirma o valor `Manual de Campanha`
2. Pagina a busca filtrada e coleta título, autor, data e handle de cada item
3. Visita a página de cada item e localiza o link do PDF (*bitstream*)
4. Remove duplicatas e classifica os documentos revogados
5. Baixa os PDFs — vigentes em `Arquivos/`, revogados em `Arquivos/Revogados/`
6. Salva o catálogo completo em `webscraping/Coleta/manuais_campanha.json`

**Sobre o acesso:** a coleta usa a interface pública de busca do DSpace, com uma
requisição por segundo e `User-Agent` identificável. O intervalo é proposital —
são centenas de requisições a um servidor institucional.

> **Nota:** Este notebook também está disponível como script executável em `webscraping/webscraping.py`.


## 1. Configuração

Todos os caminhos são ancorados na **raiz do projeto** (a pasta que contém o `.git`),
não no diretório do notebook. Assim este arquivo pode ser movido para qualquer
subpasta sem quebrar os caminhos de saída.


In [ ]:
import json
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, unquote

import requests
from bs4 import BeautifulSoup


# ── Host e parâmetros de busca ────────────────────────────────────────────
URL = "https://bdex.eb.mil.br"
RPP = 100                      # resultados por página (máximo aceito pelo DSpace)
PAUSA = 1.0                    # segundos entre requisições

PARAMS_MANUAL = {              # formato para APLICAR um filtro novo
    "filterquery": "Manual de Campanha",
    "filtername": "typeArchive",
    "filtertype": "equals",
}

PARAMS_BUSCA = {               # formato do filtro JÁ ATIVO (usado na paginação)
    "query": "",
    "filter_field_1": "typeArchive",
    "filter_type_1": "equals",
    "filter_value_1": "Manual de Campanha",
    "sort_by": "score",
    "order": "desc",
    "rpp": RPP,
    "etal": 0,
}


# ── Caminhos ancorados na raiz do projeto ─────────────────────────────────
def achar_raiz(marcador=".git"):
    """Sobe na árvore de diretórios até encontrar a pasta que contém o marcador."""
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / marcador).exists():
            return pasta
    return atual.parent


RAIZ = achar_raiz()
PASTA_PDF = RAIZ / "Arquivos"
PASTA_REVOGADOS = PASTA_PDF / "Revogados"
PASTA_COLETA = RAIZ / "webscraping" / "Coleta"
ARQUIVO_JSON = PASTA_COLETA / "manuais_campanha.json"

for pasta in (PASTA_PDF, PASTA_REVOGADOS, PASTA_COLETA):
    pasta.mkdir(parents=True, exist_ok=True)

print(f"raiz      : {RAIZ}")
print(f"vigentes  : {PASTA_PDF}")
print(f"revogados : {PASTA_REVOGADOS}")
print(f"catálogo  : {ARQUIVO_JSON}")

## 2. Sessão HTTP


In [ ]:
sessao = requests.Session()
sessao.headers.update({
    "User-Agent": "Pesquisa-Academica"
})

## 3. Funções auxiliares


In [ ]:
def sem_acento(texto):
    """Remove acentos para comparação insensível a acentuação."""
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


def ano_de(doc):
    """Ano do documento como int. Devolve 0 quando ausente ou inválido."""
    ano = (doc.get("data") or "")[:4]
    return int(ano) if ano.isdigit() else 0


RE_INVALIDOS = re.compile(r'[<>:"/\\|?*\x00-\x1f]')


def nome_seguro(nome, max_len=150):
    """Decodifica o percent-encoding e remove caracteres inválidos."""
    nome = unquote(nome)
    nome = RE_INVALIDOS.sub("_", nome)
    nome = re.sub(r"\s+", " ", nome).strip()
    nome = re.sub(r"\s+\.([a-zA-Z0-9]{2,5})$", r".\1", nome)

    if len(nome) > max_len:
        base, _, ext = nome.rpartition(".")
        if base and len(ext) <= 5:
            nome = base[: max_len - len(ext) - 1].rstrip() + "." + ext
        else:
            nome = nome[:max_len]

    return nome or "sem_nome.pdf"


RE_REVOGADO = re.compile(r"(?<![A-Z])REVOGAD[OA]S?(?![A-Z])")


def esta_revogado(doc):
    """Procura a marca de revogação no nome do arquivo e no título."""
    for campo in (doc.get("nome_arquivo") or "", doc.get("titulo") or ""):
        if RE_REVOGADO.search(sem_acento(campo).upper()):
            return True
    return False


print("Funções auxiliares carregadas.")

## 4. Funções de coleta


In [ ]:
def ler_facet_tipos(pagina=0):
    """Lê uma página do facet 'Tipo de Documento'."""
    r = sessao.get(f"{URL}/jspui/simple-search",
                   params={"query": "", "typeArchive_page": pagina},
                   timeout=60)
    r.raise_for_status()
    r.encoding = "utf-8"

    bloco = BeautifulSoup(r.text, "html.parser").find("div", id="facet_typeArchive")
    if bloco is None:
        return []

    resultado = []
    for li in bloco.find_all("li", class_="list-group-item"):
        link = li.find("a")
        badge = li.find("span", class_="badge")
        if link is None or badge is None:
            continue
        params = parse_qs(urlparse(link["href"]).query)
        valor = params.get("filterquery", [None])[0]
        if valor is None:
            continue
        resultado.append((
            link.get_text(strip=True),
            valor,
            int(badge.get_text(strip=True)),
        ))
    return resultado


RE_TOTAL = re.compile(r"^Resultado \d+-\d+ de (\d+)")


def contar_total(soup):
    """Extrai o total de resultados do texto 'Resultado 1-10 de 273.'"""
    for txt in soup.stripped_strings:
        m = RE_TOTAL.match(txt)
        if m:
            return int(m.group(1))
    return None


def extrair_resultados(soup):
    """Extrai os documentos de uma página de resultados."""
    tabela = soup.find("table", class_="table")
    if tabela is None:
        return []

    docs = []
    for tr in tabela.find_all("tr"):
        td_titulo = tr.find("td", attrs={"headers": "t2"})
        if td_titulo is None:
            continue
        link = td_titulo.find("a")
        if link is None:
            continue
        td_data = tr.find("td", attrs={"headers": "t1"})
        td_autor = tr.find("td", attrs={"headers": "t3"})
        docs.append({
            "data": td_data.get_text(strip=True) if td_data else None,
            "titulo": link.get_text(strip=True),
            "autor": td_autor.get_text(strip=True) if td_autor else None,
            "handle": link["href"],
        })
    return docs


def achar_pdf(handle):
    """Dado o handle de um item, devolve (url_pdf, nome_arquivo) ou (None, None)."""
    url_item = urljoin(URL, handle)
    r = sessao.get(url_item, timeout=60)
    r.raise_for_status()
    r.encoding = "utf-8"

    soup = BeautifulSoup(r.text, "html.parser")
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/bitstream/" in href and href.lower().endswith(".pdf"):
            url_pdf = urljoin(url_item, href)
            nome = nome_seguro(href.rsplit("/", 1)[-1])
            return url_pdf, nome
    return None, None


def baixar(url_pdf, nome_arquivo, pasta_destino):
    """Baixa um PDF. Devolve (caminho, status); caminho None indica falha."""
    destino = pasta_destino / nome_arquivo
    if destino.exists() and destino.stat().st_size > 0:
        return destino, "já existe"

    with sessao.get(url_pdf, timeout=300, stream=True) as resp:
        resp.raise_for_status()
        tipo = resp.headers.get("content-type", "")
        if "pdf" not in tipo.lower():
            return None, f"não é PDF ({tipo})"
        temp = destino.with_name(destino.name + ".parcial")
        with open(temp, "wb") as f:
            for pedaco in resp.iter_content(chunk_size=65536):
                f.write(pedaco)
        temp.replace(destino)

    return destino, f"{destino.stat().st_size / 1024 / 1024:.1f} MB"


print("Funções de coleta carregadas.")